# Lab 3 · Train & compare disengagement models (MLflow)

Build a small **ML pipeline** that trains three endpoint-servable model families — **Logistic Regression**, **LightGBM**, **XGBoost** — each with **hyperparameter tuning**. Every run is tracked in an **MLflow experiment** so you can compare them, then we register the winner for real-time serving.

**Target:** `is_disengaged` from `gold.resident_360`.
**Note:** we deliberately **exclude** the three columns that *define* the label (`avg_daily_steps`, `events_attended`, `programmes_dropped`) so the models learn genuine risk signals instead of memorising the rule.

> **Attach** the `lh_resident360` Lakehouse first.

## 1. Dependencies (LightGBM / XGBoost ship with the Fabric ML runtime; install if missing)

In [ ]:
try:
    import lightgbm, xgboost  # noqa
except Exception:
    %pip install -q lightgbm xgboost
    import lightgbm, xgboost  # noqa
print("lightgbm", lightgbm.__version__, "| xgboost", xgboost.__version__)

## 2. Load features + label

In [ ]:
import numpy as np, pandas as pd, mlflow
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

pdf = spark.table("gold.resident_360").toPandas()

# ordinal-encode screening risk
risk_map = {"Not Screened": 0, "Low": 1, "Moderate": 2, "High": 3}
pdf["screening_risk_ord"] = pdf["screening_risk"].map(risk_map).fillna(0)

FEATURES = ['avg_mvpa_min','avg_sleep_min','days_goal_met','active_days','meal_logs',
            'avg_calories','pct_healthier_choice','events_booked','programmes_enrolled',
            'healthpoints_earned','healthpoints_redeemed','vouchers_redeemed','voucher_value_sgd',
            'challenges_active','avg_challenge_progress','latest_bmi','latest_systolic',
            'screening_risk_ord']
LABEL = "is_disengaged"

X = pdf[FEATURES].astype(float).fillna(0.0)
y = pdf[LABEL].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("train:", X_train.shape, "| positives:", int(y_train.sum()), "/", len(y_train))

## 3. Train + tune three model families — one MLflow run per configuration
Each run logs its **params** and **metrics** (ROC-AUC, accuracy, F1) to the experiment.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

mlflow.set_experiment("resident360-disengagement")
mlflow.autolog(disable=True)

def evaluate(model, name, params):
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        preds = (proba >= 0.5).astype(int)
        metrics = {'auc': roc_auc_score(y_test, proba),
                   'accuracy': accuracy_score(y_test, preds),
                   'f1': f1_score(y_test, preds, zero_division=0)}
        mlflow.log_params(params)
        mlflow.log_param('model_family', name.split('_')[0])
        mlflow.log_metrics(metrics)
        print(f"{name:22s} AUC={metrics['auc']:.4f}  ACC={metrics['accuracy']:.4f}  F1={metrics['f1']:.4f}")
        return dict(name=name, family=name.split('_')[0], model=model, **metrics, params=params)

runs = []
# Logistic Regression — tune C
for C in [0.1, 1.0, 3.0]:
    m = Pipeline([('scale', StandardScaler()),
                  ('clf', LogisticRegression(C=C, max_iter=1000, class_weight='balanced'))])
    runs.append(evaluate(m, f'logreg_C{C}', {'C': C}))
# LightGBM — tune n_estimators / num_leaves
for n, leaves in [(200, 31), (400, 63)]:
    m = LGBMClassifier(n_estimators=n, num_leaves=leaves, learning_rate=0.05,
                       class_weight='balanced', random_state=42, verbose=-1)
    runs.append(evaluate(m, f'lightgbm_n{n}_l{leaves}', {'n_estimators': n, 'num_leaves': leaves}))
# XGBoost — tune n_estimators / max_depth
for n, depth in [(200, 4), (400, 6)]:
    spw = float((y_train == 0).sum()) / max(1, int((y_train == 1).sum()))
    m = XGBClassifier(n_estimators=n, max_depth=depth, learning_rate=0.05,
                      subsample=0.9, scale_pos_weight=spw, eval_metric='logloss', random_state=42)
    runs.append(evaluate(m, f'xgboost_n{n}_d{depth}', {'n_estimators': n, 'max_depth': depth}))

## 4. Pick the winner (highest ROC-AUC)
Open **Experiments → resident360-disengagement** in the workspace to compare all runs visually.

In [ ]:
best = max(runs, key=lambda r: r['auc'])
print('WINNER:', best['name'], '| AUC =', round(best['auc'], 4), '| family =', best['family'])

## 5. Register the winner as a **native flavor** with a scalar signature
Native flavor + scalar in/out is what makes the model **servable on a real-time endpoint**.

In [ ]:
from mlflow.models.signature import infer_signature
pred_out = pd.DataFrame({'prediction': best['model'].predict(X_train).astype(int)})
signature = infer_signature(X_train, pred_out)   # column-based in AND out (endpoint-servable)
input_example = X_train.head(3)
REGISTERED = 'resident360_disengagement'

with mlflow.start_run(run_name=f"register_{best['name']}"):
    mlflow.log_params(best['params'])
    mlflow.log_metric('auc', best['auc'])
    if best['family'] == 'lightgbm':
        mlflow.lightgbm.log_model(best['model'], 'model', signature=signature,
                                  input_example=input_example, registered_model_name=REGISTERED)
    elif best['family'] == 'xgboost':
        mlflow.xgboost.log_model(best['model'], 'model', signature=signature,
                                 input_example=input_example, registered_model_name=REGISTERED)
    else:
        mlflow.sklearn.log_model(best['model'], 'model', signature=signature,
                                 input_example=input_example, registered_model_name=REGISTERED)
print('Registered model:', REGISTERED)

## ✅ Next
1. Open the **`resident360_disengagement`** model → select **Version 1** → **Activate version endpoint** from the Home ribbon (requires the *ML model endpoints (Preview)* tenant setting).
2. In **Manage endpoints**, set **Default version = Version 1** and copy the `/score` endpoint URL.
3. Run **`09_call_model_endpoint.ipynb`** to score sample residents from the registry and via the live endpoint.
